# Limpeza e Validação — Code49 / Claudio Alphaville Tamboré

Pipeline: Raw JSON → Code49Adapter (CanonicalRecord) → Pipeline stages → Silver schema validation.

**Entrada:** `data/code49_claudio_raw.json` (gerado pelo 01_explore)
**Saída:** `output/code49_claudio_clean.csv`

In [2]:
import datawork
datawork.setup()

import json
from pathlib import Path

import pandas as pd

from datawork.display import show_sample, show_stats
from datawork.profiling import completeness_report

In [3]:
# === STEP 1: Carregar raw data ===
RAW_PATH = Path("../../data/code49_claudio_raw.json")
CLEAN_OUTPUT = Path("../../output/code49_claudio_clean.csv")

with open(RAW_PATH, encoding="utf-8") as f:
    raw_records = json.load(f)

print(f"Carregados {len(raw_records)} registros raw de {RAW_PATH}")

Carregados 25 registros raw de ..\..\data\code49_claudio_raw.json


In [4]:
# === STEP 2: Transformar via Code49Adapter → CanonicalRecord ===
from petrus.infrastructure.mdm.connectors.code49_adapter import Code49Adapter
from petrus.domain.entities.mdm_types import CanonicalRecord
import dataclasses

adapter = Code49Adapter()

canonical_records = []
errors = []

for i, raw in enumerate(raw_records):
    try:
        record = adapter.transform(raw)
        canonical_records.append(dataclasses.asdict(record))
    except Exception as e:
        errors.append(f"  [{i}] source_id={raw.get('source_id', '?')}: {e}")

print(f"Transformados: {len(canonical_records)}/{len(raw_records)}")
if errors:
    print(f"Erros ({len(errors)}):")
    for e in errors:
        print(e)

# Converter para DataFrame
df_clean = pd.DataFrame(canonical_records)
print(f"\nDataFrame: {df_clean.shape[0]} rows x {df_clean.shape[1]} cols")

Transformados: 25/25

DataFrame: 25 rows x 42 cols


In [5]:
# === STEP 3: Analisar resultado do adapter ===
show_sample(df_clean, n=5, title="Dados após Code49Adapter")

# Campos chave do CanonicalRecord
campos_chave = [
    "titulo", "cidade", "uf", "bairro",
    "area_total_m2", "area_construida_m2", "pe_direito_m",
    "numero_docas", "vagas_estacionamento", "potencia_eletrica_kva",
    "valor_locacao", "valor_venda", "tipo_operacao", "tipo",
    "source_id", "source_url",
]

print("\n=== Campos chave ===")
for col in campos_chave:
    if col in df_clean.columns:
        non_null = df_clean[col].notna().sum()
        total = len(df_clean)
        pct = non_null / total * 100
        print(f"  {col:30s}: {non_null:3d}/{total} ({pct:5.1f}%)")
    else:
        print(f"  {col:30s}: COLUNA AUSENTE")


  Dados após Code49Adapter
Shape: 25 rows x 42 cols


,endereco,logradouro,numero,complemento,unidade,bairro,cidade,uf,cep,latitude,longitude,area_total_m2,area_construida_m2,area_piso_m2,area_escritorio_m2,area_mezanino_m2,pe_direito_m,numero_docas,vagas_estacionamento,potencia_eletrica_kva,elevador,gerador,ponte_rolante,zoneamento,titulo,tipo,categoria,tipo_operacao,valor_locacao,valor_venda,valor_condominio,iptu_valor,status,inquilino,proprietario_nome,proprietario_telefone,proprietario_email,observacoes,dados_extras,source_url,source_id,data_coleta
0,None,None,None,None,None,Brás,São Paulo,SP,None,None,None,1458.00,1317.73,None,None,None,8.0,1.0,6.0,300.0,None,None,None,None,GALPÃO COM RENDA A VENDA EM SOROCABA CONDOMÍNI...,galpao,None,venda,NaN,4280000.0,None,None,None,None,None,None,None,GALPÃO COM RENDA A VENDA EM SOROCABA CONDOMÍNI...,{'imagens_fonte': ['https://www.claudioalphavi...,https://www.claudioalphavilletambore.com.br/91...,918,2026-08-03 18:01:24.467945
1,None,None,None,None,None,Alphaville Industrial,Barueri,SP,None,None,None,827.23,827.23,None,None,None,NaN,NaN,8.0,NaN,None,None,None,None,GALPÃO NOVO EM ALPHAVILLE PARA LOCAÇÃO ELEVADO...,galpao,None,locacao,NaN,NaN,None,None,None,None,None,None,None,GALPÃO COMERCIAL PARA LOCAÇÃO EM ALPHAVILLE PR...,{'imagens_fonte': ['https://www.claudioalphavi...,https://www.claudioalphavilletambore.com.br/90...,905,2026-08-03 18:01:24.468134
2,None,None,None,None,None,Alphaville Empresarial,Barueri,SP,None,None,None,654.00,654.00,None,None,None,7.0,NaN,6.0,NaN,None,None,None,None,GALPÃO LOCAÇÃO ALPHAVILLE ALAMEDA JURA PÉ DIRE...,galpao,None,locacao,16000.0,NaN,None,None,None,None,None,None,None,"GALPÃO LOCAÇÃO PROXIMO AS SAÍDAS, LOCALIZADO N...",{'imagens_fonte': ['https://www.claudioalphavi...,https://www.claudioalphavilletambore.com.br/89...,897,2026-08-03 18:01:24.468245
3,None,None,None,None,None,Alphaville Industrial,Barueri,SP,None,None,None,600.00,500.00,None,None,None,NaN,NaN,10.0,NaN,None,None,None,None,Galpão Para Venda Ou Locação Em Alphaville Ala...,galpao,None,venda,NaN,NaN,None,None,None,None,None,None,None,Galpão/Depósito/Armazém e 4 banheiros para Alu...,{'imagens_fonte': ['https://www.claudioalphavi...,https://www.claudioalphavilletambore.com.br/87...,879,2026-08-03 18:01:24.468325
4,None,None,None,None,None,Alphaville,Santana de Parnaíba,SP,None,None,None,5315.00,NaN,None,None,None,NaN,NaN,114.0,NaN,None,None,None,None,Galpão para Locação em Alphaville Tamboré Com ...,galpao,None,locacao,NaN,NaN,None,None,None,None,None,None,None,Galpão Prédio Comercial no Polo Empresarial Ta...,{'imagens_fonte': ['https://www.claudioalphavi...,https://www.claudioalphavilletambore.com.br/80...,805,2026-08-03 18:01:24.468396



=== Campos chave ===
  titulo                        :  25/25 (100.0%)
  cidade                        :  25/25 (100.0%)
  uf                            :  25/25 (100.0%)
  bairro                        :  25/25 (100.0%)
  area_total_m2                 :  16/25 ( 64.0%)
  area_construida_m2            :  14/25 ( 56.0%)
  pe_direito_m                  :   6/25 ( 24.0%)
  numero_docas                  :   5/25 ( 20.0%)
  vagas_estacionamento          :  14/25 ( 56.0%)
  potencia_eletrica_kva         :   1/25 (  4.0%)
  valor_locacao                 :  13/25 ( 52.0%)
  valor_venda                   :   7/25 ( 28.0%)
  tipo_operacao                 :  25/25 (100.0%)
  tipo                          :  25/25 (100.0%)
  source_id                     :  25/25 (100.0%)
  source_url                    :  25/25 (100.0%)


In [6]:
# === STEP 4: Limpeza — tratar anomalias ===
df_limpo = df_clean.copy()

# 4a. Remover precos placeholder (> R$ 100M venda, > R$ 1M locacao)
mask_sale = df_limpo["valor_venda"].notna() & (df_limpo["valor_venda"] > 100_000_000)
mask_rent = df_limpo["valor_locacao"].notna() & (df_limpo["valor_locacao"] > 1_000_000)
n_removed_sale = mask_sale.sum()
n_removed_rent = mask_rent.sum()
df_limpo.loc[mask_sale, "valor_venda"] = None
df_limpo.loc[mask_rent, "valor_locacao"] = None
print(f"Precos placeholder removidos: {n_removed_sale} venda, {n_removed_rent} locacao")

# 4b. Recalcular tipo_operacao apos limpeza
def recalc_operacao(row):
    has_rent = pd.notna(row.get("valor_locacao"))
    has_sale = pd.notna(row.get("valor_venda"))
    if has_rent and has_sale:
        return "ambos"
    if has_sale:
        return "venda"
    if has_rent:
        return "locacao"
    return row.get("tipo_operacao")

df_limpo["tipo_operacao"] = df_limpo.apply(recalc_operacao, axis=1)

# 4c. Stats apos limpeza
print(f"\nApos limpeza: {len(df_limpo)} registros")
show_stats(df_limpo)

Precos placeholder removidos: 0 venda, 1 locacao

Apos limpeza: 25 registros

Total registros: 25

Campos numéricos:
       area_total_m2  area_construida_m2  area_piso_m2  valor_locacao   valor_venda  pe_direito_m
count          16.00           14.000000           0.0           12.0  7.000000e+00      6.000000
mean         4401.23         3494.006429           NaN        55425.0  9.125714e+06      9.333333
min           600.00          500.000000           NaN        10000.0  1.600000e+06      7.000000
max         21653.00        12215.830000           NaN       300000.0  2.500000e+07     12.000000

cidade:
  Barueri: 23
  São Paulo: 1
  Santana de Parnaíba: 1

tipo_operacao:
  locacao: 17
  venda: 8

tipo:
  galpao: 25

status:
  None: 25


In [7]:
# === STEP 5: Validar contra Silver schema ===
# O schema pode falhar em campos que nao temos — isso e esperado
# Usamos lazy=True para ver TODOS os erros de uma vez
from datawork.contracts.silver import CleanRecordSchema

try:
    CleanRecordSchema.validate(df_limpo, lazy=True)
    print("Silver schema validation PASSED")
except Exception as e:
    print(f"Silver schema validation FAILED (esperado no primeiro run):")
    print(f"  {e}")
    print("\nIsso e normal — nem todos os campos sao obrigatorios.")

Silver schema validation PASSED


In [8]:
# === STEP 6: Completude final e export ===
print("=== Completude final ===")
display(completeness_report(df_limpo))

# Exportar CSV limpo
from datawork.io.pushers import export_csv

CLEAN_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
export_csv(df_limpo, CLEAN_OUTPUT)

# Tambem salvar como JSON para uso futuro
clean_json = CLEAN_OUTPUT.with_suffix(".json")
from datawork.io.pushers import to_canonical_records
records = to_canonical_records(df_limpo)
with open(clean_json, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2, default=str)
print(f"JSON limpo salvo em: {clean_json}")

=== Completude final ===


,column,non_null,non_empty,pct_filled
0,cidade,25,25,100.0
1,bairro,25,25,100.0
2,uf,25,25,100.0
3,source_url,25,25,100.0
4,tipo_operacao,25,25,100.0
5,data_coleta,25,25,100.0
6,dados_extras,25,25,100.0
7,tipo,25,25,100.0
8,source_id,25,25,100.0
9,titulo,25,24,96.0


Exported 25 rows to ..\..\output\code49_claudio_clean.csv
JSON limpo salvo em: ..\..\output\code49_claudio_clean.json


## Próximos Passos

- Revisar anomalias acima e ajustar thresholds se necessário
- Quando API estiver rodando: descomentar push via `push_clean_to_api`
- Iterar: se campos estão faltando, voltar ao scraper e melhorar parsing